# Semantic Model & Report Analyzer

This notebook scans a Power BI **semantic model** and its **reports** in a Fabric workspace using
[`semantic-link`](https://learn.microsoft.com/fabric/data-science/semantic-link-overview) (`sempy`)
and [`semantic-link-labs`](https://github.com/microsoft/semantic-link-labs) (`sempy_labs`).

## ⚠️ Prerequisite: Reports must be in PBIP / PBIR format

Detailed visual / filter / conditional-formatting analysis (Section 5 onward) requires reports
to be saved in the **enhanced report format (PBIR)**, typically as part of a **PBIP (Power BI Project)**.
Standard `.pbix` reports do **not** expose the JSON layout that `ReportWrapper` needs, and those
reports will be skipped during scanning.

**To convert a report to PBIP / PBIR:**

1. Open the report in **Power BI Desktop**.
2. Go to **File → Options and settings → Options → Preview features**.
3. Enable **"Power BI Project (.pbip) save option"** and **"Store reports using enhanced metadata format (PBIR)"**.
4. Click **File → Save as** and choose **Power BI project files (*.pbip)**.
5. Publish (or upload) the resulting PBIP/PBIR report to the Fabric workspace.

More info:
- PBIP overview: https://learn.microsoft.com/power-bi/developer/projects/projects-overview
- PBIR format: https://powerbi.microsoft.com/blog/power-bi-enhanced-report-format-pbir-in-power-bi-desktop-developer-mode-preview/

---

It answers:

1. **Which measures / columns are actually "alive"** in visuals, filters, and conditional formatting?
2. **Unused columns & measures** in the semantic model.
3. **Where is a given column/measure used** (lineage across model + reports)?
4. **Dependency tree** for measures (measure → measure → column).
5. **Export documentation** for the model and the reports.

> Run cells top-to-bottom. The notebook is parameterized — set `WORKSPACE` and `DATASET` in the
> *Parameters* cell. Reports under that workspace bound to the dataset are auto-discovered.


## 1. Install & import dependencies

`sempy` is preinstalled on Fabric Spark. `semantic-link-labs` adds report/model utilities
(report wrapper, dependency tree, unused objects, documentation).


In [ ]:
# Install semantic-link-labs (sempy_labs). Runs once per session.
%pip install -q semantic-link-labs


In [ ]:
import re
import json
import pandas as pd

import sempy.fabric as fabric
import sempy_labs as labs
import sempy_labs.report as labs_report
from sempy_labs.report import ReportWrapper

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)
print("sempy_labs version:", getattr(labs, "__version__", "unknown"))


## 2. Parameters

Set the **workspace** and **dataset (semantic model)** to analyze. Leave `WORKSPACE = None`
to use the current notebook's workspace. The notebook will auto-discover all reports in the
workspace bound to `DATASET`.


In [ ]:
# ---- EDIT THESE ----
WORKSPACE = None          # e.g. "Sales Analytics" or workspace GUID; None = current workspace
DATASET   = "MyModel"     # semantic model display name
REPORTS   = None          # None = auto-discover all reports bound to DATASET; or list of report names
EXPORT_DIR = "/lakehouse/default/Files/model_docs"   # where docs are written (must be a mounted lakehouse)
# --------------------

if WORKSPACE is None:
    WORKSPACE = fabric.resolve_workspace_name()
print(f"Workspace : {WORKSPACE}")
print(f"Dataset   : {DATASET}")


## 3. Inventory the semantic model

Pull tables, columns, measures, and relationships from the model. These are the universe of
"objects" that may or may not be used by reports.


In [ ]:
# Pull model metadata via DMV-style queries that sempy exposes.
tables_df       = fabric.list_tables(dataset=DATASET, workspace=WORKSPACE, extended=True)
columns_df      = fabric.list_columns(dataset=DATASET, workspace=WORKSPACE, extended=True)
measures_df     = fabric.list_measures(dataset=DATASET, workspace=WORKSPACE)
relationships_df = fabric.list_relationships(dataset=DATASET, workspace=WORKSPACE)

print(f"Tables        : {len(tables_df)}")
print(f"Columns       : {len(columns_df)}")
print(f"Measures      : {len(measures_df)}")
print(f"Relationships : {len(relationships_df)}")
measures_df.head()


## 4. Discover reports bound to the model

Each Power BI report stores its visuals/filters in a JSON layout. `ReportWrapper` parses that
layout and exposes the objects it actually references.


In [ ]:
# Auto-discover reports bound to DATASET in WORKSPACE.
# Safety: if REPORTS was set in a previous run, honour it; otherwise auto-discover.
# To force auto-discovery, ensure REPORTS = None in the Parameters cell and re-run it.
print(f"REPORTS parameter = {repr(REPORTS)}")
if REPORTS is not None:
    print("  ℹ️  REPORTS is set — only those reports will be scanned (auto-discovery skipped).")

all_reports = labs.list_reports_using_semantic_model(dataset=DATASET, workspace=WORKSPACE)
print(f"All reports dataframe shape : {all_reports.shape}")
print(f"Columns                     : {list(all_reports.columns)}")
display(all_reports)

# Print raw rows to confirm all entries
print("\nRaw rows:")
for idx, row in all_reports.iterrows():
    print(f"  row {idx}: {dict(row)}")

# ── Column resolution helpers ─────────────────────────────────────────────────
def _pick(df, *candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

rn_col  = _pick(all_reports, "Report Name",           "ReportName",      "Name")
ws_col  = _pick(all_reports, "Report Workspace Name", "Workspace Name",   "Report Workspace", "WorkspaceName", "Workspace")
wid_col = _pick(all_reports, "Report Workspace Id",   "Workspace Id",     "WorkspaceId",      "Report Workspace Id")
rid_col = _pick(all_reports, "Report Id",             "ReportId",         "Id")

print(f"\nResolved columns — name: {rn_col} | workspace name: {ws_col} | workspace id: {wid_col} | report id: {rid_col}")
print(f"REPORTS param value: {repr(REPORTS)}")   # must be None to auto-discover

# ── Always reset index before iterating to avoid stale-index bugs ─────────────
all_reports = all_reports.reset_index(drop=True)

# ── Build report_specs using iterrows() — immune to non-contiguous index ──────
if REPORTS is None:
    report_specs = []
    for _, row in all_reports.iterrows():                # iterrows ignores index gaps
        n   = str(row[rn_col]).strip()  if rn_col  else ""
        ws  = str(row[ws_col]).strip()  if ws_col  else ""
        wid = str(row[wid_col]).strip() if wid_col else ""
        rid = str(row[rid_col]).strip() if rid_col else ""

        if not n or n.lower() in ("nan", "none", ""):
            continue

        # Prefer workspace ID (GUID) — more reliable for ReportWrapper than display name
        ws_resolved = wid if wid and wid.lower() not in ("nan", "none", "") \
                      else (ws if ws and ws.lower() not in ("nan", "none", "") else WORKSPACE)

        report_specs.append({"name": n, "workspace": ws_resolved, "report_id": rid})
else:
    if isinstance(REPORTS, str):
        report_specs = [{"name": REPORTS, "workspace": WORKSPACE, "report_id": ""}]
    else:
        report_specs = [{"name": r, "workspace": WORKSPACE, "report_id": ""} for r in REPORTS]

# Backwards-compat alias
report_names = [s["name"] for s in report_specs]

print(f"\nFinal report_specs ({len(report_specs)}) — expected {len(all_reports)} rows:")
for i, s in enumerate(report_specs):
    print(f"  [{i}] name='{s['name']}'  workspace='{s['workspace']}'  report_id='{s['report_id']}'")

# Show as table for easy verification
display(pd.DataFrame(report_specs))

if len(report_specs) != len(all_reports):
    print(f"\n⚠️  WARNING: {len(all_reports)} reports found but only {len(report_specs)} specs built.")
    print("   Re-run the Parameters cell (cell 7) to ensure REPORTS = None, then re-run this cell.")
else:
    print(f"\n✅ All {len(report_specs)} reports captured.")


## 5. Map measures & columns to visuals, filters and conditional formatting

For every report we walk its visual layer and collect every object reference. `ReportWrapper`
methods like `list_visual_objects`, `list_report_filters`, `list_page_filters`,
`list_visual_filters` and `list_visual_calc_items` each return DataFrames that include the
table/column/measure references. We union them into one **usage map**.


In [ ]:
import traceback

def _norm(df: pd.DataFrame, report: str, surface: str) -> pd.DataFrame:
    """Normalize a ReportWrapper dataframe to (report, surface, page, visual, table, object, kind)."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["Report", "Surface", "Page", "Visual", "Table", "Object", "Kind"])
    out = pd.DataFrame()
    out["Report"]  = [report] * len(df)
    out["Surface"] = surface
    out["Page"]    = df.get("Page Display Name", df.get("Page Name", pd.Series([None]*len(df)))).values
    out["Visual"]  = df.get("Visual Name", df.get("Visual ID", pd.Series([None]*len(df)))).values
    out["Table"]   = df.get("Table Name", pd.Series([None]*len(df))).values
    out["Object"]  = df.get("Object Name",
                            df.get("Measure Name",
                            df.get("Column Name",
                            df.get("Hierarchy Name", pd.Series([None]*len(df)))))).values
    out["Kind"]    = df.get("Object Type",
                            df.get("Type", pd.Series(["Unknown"]*len(df)))).values
    return out

VERBOSE_ERRORS = True  # set False to suppress tracebacks

usage_frames = []
pbir_required_reports = []
scanned_reports = []
skipped_reports = []

print(f"report_specs has {len(report_specs)} entries — starting scan...\n")

for spec in report_specs:
    r   = spec["name"]
    ws  = spec["workspace"]
    rid = spec.get("report_id", "")
    print(f"\n{'─'*60}")
    print(f"Scanning: '{r}'  workspace='{ws}'  report_id='{rid}'")

    rw = None
    last_err = Exception("no attempt made")

    # Build attempts: workspace ID preferred over display name; also try report ID if available
    attempts = []
    if ws:
        attempts.append({"report": r, "workspace": ws})
    if rid:
        attempts.append({"report": rid, "workspace": ws})   # try by report GUID
    if WORKSPACE != ws:
        attempts.append({"report": r, "workspace": WORKSPACE})
    attempts.append({"report": r})  # last resort: no workspace arg

    # Deduplicate while preserving order
    seen_keys, unique_attempts = [], []
    for a in attempts:
        key = tuple(sorted(a.items()))
        if key not in seen_keys:
            seen_keys.append(key)
            unique_attempts.append(a)

    for attempt in unique_attempts:
        try:
            rw = ReportWrapper(**attempt)
            print(f"  ✓ Opened with {attempt}")
            break
        except Exception as e:
            last_err = e
            if VERBOSE_ERRORS:
                print(f"  [tried {attempt}] {type(e).__name__}: {str(e)[:150]}")

    if rw is None:
        msg = str(last_err)
        if "PBIR format" in msg or "pbir" in msg.lower():
            print(f"  ⚠️  Not in PBIR format — skipping visual analysis.")
            pbir_required_reports.append(r)
        else:
            print(f"  ❌ Could not open: {msg}")
            skipped_reports.append(r)
        continue

    report_had_data = False
    pbir_errors = []
    other_errors = []

    surfaces = {
        "Visual"          : "list_visual_objects",
        "Report Filter"   : "list_report_filters",
        "Page Filter"     : "list_page_filters",
        "Visual Filter"   : "list_visual_filters",
        "Visual Calc"     : "list_visual_calc_items",
    }
    for surface, method in surfaces.items():
        fn = getattr(rw, method, None)
        if fn is None:
            continue
        try:
            result = fn()
            normalized = _norm(result, r, surface)
            if len(normalized) > 0:
                usage_frames.append(normalized)
                report_had_data = True
                print(f"  ✓ {surface}: {len(normalized)} references")
            else:
                print(f"  · {surface}: (empty)")
        except Exception as e:
            error_msg = str(e)
            if "PBIR format" in error_msg or "pbir" in error_msg.lower():
                pbir_errors.append(surface)
            else:
                other_errors.append(f"{surface}: {type(e).__name__}: {error_msg[:120]}")
                if VERBOSE_ERRORS:
                    traceback.print_exc(limit=2)

    if pbir_errors:
        print(f"  ⚠️  PBIR required for: {', '.join(pbir_errors)}")
        pbir_required_reports.append(r)
    elif report_had_data:
        print(f"  ✅ Successfully scanned")
        scanned_reports.append(r)
    else:
        if other_errors:
            print(f"  ⚠️  Methods failed:")
            for err in other_errors[:5]:
                print(f"     • {err}")
        else:
            print(f"  ⚠️  No data (report may be empty)")
        skipped_reports.append(r)

usage_df = pd.concat(usage_frames, ignore_index=True) if usage_frames else \
           pd.DataFrame(columns=["Report","Surface","Page","Visual","Table","Object","Kind"])
usage_df = usage_df.dropna(subset=["Object"]).drop_duplicates().reset_index(drop=True)

print(f"\n{'='*60}")
print(f"Summary:")
print(f"  Total specs:            {len(report_specs)}")
print(f"  Successfully scanned:   {len(scanned_reports)}")
print(f"  Requires PBIR format:   {len(pbir_required_reports)}")
print(f"  Skipped (other errors): {len(skipped_reports)}")
print(f"  Object references:      {len(usage_df)}")
print(f"{'='*60}")

if pbir_required_reports:
    print(f"\n📌 Reports needing PBIR format: {pbir_required_reports}")
    print("   See the Prerequisites section at the top of this notebook for conversion steps.")

usage_df.head(20)


## 6. "Alive" measures — actually referenced by the report layer

A measure is **alive** if it appears anywhere in the usage map (visual fields, filters, or
conditional formatting expressions). Everything else is a candidate for removal.


In [ ]:
# Build the set of (Table, Object) used directly by reports (visuals / filters / CF)
report_refs = set(zip(usage_df["Table"].astype(str), usage_df["Object"].astype(str)))

# Pull full measure-to-measure / measure-to-column dependency graph
# sempy_labs >= 0.8 deprecated dataset/workspace params — try both calling patterns
try:
    deps_df = labs.get_measure_dependencies(dataset=DATASET, workspace=WORKSPACE)
except TypeError:
    deps_df = labs.get_measure_dependencies()
display(deps_df.head())

# ── adjacency maps ─────────────────────────────────────────────────────────────
# parent_to_children : measure M  → set of (table, object) that M's DAX references
# child_to_parents   : object  O  → set of (table, measure) whose DAX references O
parent_to_children = {}
child_to_parents   = {}

for _, row in deps_df.iterrows():
    parent_key = (row.get("Table Name"),       row.get("Object Name"))
    child_key  = (row.get("Referenced Table"), row.get("Referenced Object"))
    parent_to_children.setdefault(parent_key, set()).add(child_key)
    child_to_parents.setdefault(child_key,   set()).add(parent_key)

# ── closure helpers ────────────────────────────────────────────────────────────
def expand_down(seed: set) -> set:
    """Walk DOWN (parent→child): find everything alive measures pull in."""
    seen  = set(seed)
    stack = list(seed)
    while stack:
        node = stack.pop()
        for child in parent_to_children.get(node, ()):
            if child not in seen:
                seen.add(child)
                stack.append(child)
    return seen

def expand_up(seed: set) -> set:
    """Walk UP (child→parent): find every measure that transitively uses alive nodes."""
    seen  = set(seed)
    stack = list(seed)
    while stack:
        node = stack.pop()
        for parent in child_to_parents.get(node, ()):
            if parent not in seen:
                seen.add(parent)
                stack.append(parent)
    return seen

# ── build the "alive" set ──────────────────────────────────────────────────────
# Seed = objects directly referenced in reports
# Step 1: expand UP  → find all measures transitively consumed by the report layer
# Step 2: expand DOWN → find all columns/objects those measures pull in via DAX
alive_step1 = expand_up(report_refs)
alive       = expand_down(alive_step1)

measures_keys  = set(zip(measures_df["Table Name"], measures_df["Measure Name"]))
alive_measures = measures_keys & alive
unused_measures = measures_keys - alive

direct_in_report = report_refs & measures_keys
via_chain        = alive_measures - direct_in_report

print(f"Measures total            : {len(measures_keys)}")
print(f"Alive measures            : {len(alive_measures)}")
print(f"  → directly in report    : {len(direct_in_report)}")
print(f"  → alive via chain only  : {len(via_chain)}")
print(f"Unused measures           : {len(unused_measures)}")


## 6b. Top most-used DAX measures

Ranks every measure by two combined signals:

| Signal | Source | Available without PBIR? |
|---|---|---|
| **Report References** | `usage_df` — how many distinct visual/filter slots reference this measure across all reports | Only with PBIR |
| **DAX Callers** | `child_to_parents` — how many other measures call this measure inside their DAX expressions | ✅ Always |

### How the score works

```
Total Usage Score = Report References + DAX Callers
```

- **Report References** counts every distinct slot (visual field, report filter, page filter, visual filter, conditional formatting) in which the measure appears — higher means more visible to end users.
- **DAX Callers** counts how many *other measures* reference this measure inside their DAX formula — higher means this is a core "helper" measure depended on by the rest of the model.
- A measure with a high **Report References** score is directly consumed by visuals.
- A measure with a high **DAX Callers** score is a foundational building block — removing it would break many other measures.
- A measure with a high score on **both** is critical: widely used in reports *and* heavily relied on in the model.
- A measure with a score of **0** on both is a candidate for cleanup (also flagged in the unused section below).

> **Note:** When no PBIR reports are available, Report References will be 0 for all measures and the ranking reflects model-internal reuse only (DAX Callers).


In [ ]:
TOP_N = 100  # change to 50 if you prefer a shorter list

# ── Signal 1: report-layer hit count ─────────────────────────────────────────
# Count distinct (Report, Page, Visual, Surface) slots per measure across all reports.
if not usage_df.empty:
    rpt_counts = (
        usage_df[usage_df["Kind"].str.lower() == "measure"]
        .groupby(["Table", "Object"])
        .size()
        .reset_index(name="Report References")
    )
else:
    rpt_counts = pd.DataFrame(columns=["Table", "Object", "Report References"])
    print("ℹ️  No PBIR reports scanned — Report References will be 0 for all measures.")

# ── Signal 2: how many other measures call this measure in their DAX ──────────
dep_counts = {
    (ref_t, ref_o): len(callers)
    for (ref_t, ref_o), callers in child_to_parents.items()
    if (ref_t, ref_o) in measures_keys          # only count objects that are measures
}
dep_df = pd.DataFrame(
    [(t, o, c) for (t, o), c in dep_counts.items()],
    columns=["Table", "Object", "DAX Callers"],
)

# ── Build ranking table ───────────────────────────────────────────────────────
all_m = pd.DataFrame(list(measures_keys), columns=["Table", "Object"])

top_df = (
    all_m
    .merge(rpt_counts, on=["Table", "Object"], how="left")
    .merge(dep_df,     on=["Table", "Object"], how="left")
    .fillna(0)
)
top_df["Report References"] = top_df["Report References"].astype(int)
top_df["DAX Callers"]       = top_df["DAX Callers"].astype(int)
top_df["Total Usage Score"] = top_df["Report References"] + top_df["DAX Callers"]
top_df["Alive"]             = top_df.apply(
    lambda r: "✅" if (r["Table"], r["Object"]) in alive_measures else "❌", axis=1
)

top_df = (
    top_df
    .sort_values("Total Usage Score", ascending=False)
    .reset_index(drop=True)
)
top_df.index += 1   # 1-based rank

print(f"Top {min(TOP_N, len(top_df))} most-used measures  (of {len(top_df)} total)")
display(top_df.head(TOP_N))


## 7. Unused columns & measures

Identifies objects that are not reachable from any report visual, filter, or conditional formatting reference.

- **Report-aware unused** — measures/columns not reached through the report→measure→column dependency chain.
- **Graph-only fallback** — when no PBIR reports are available, measures not referenced by any other measure or relationship are flagged.


In [ ]:
# ── Resolve column names defensively (sempy versions vary) ───────────────────
def _pick_col(df, *candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

m_table = _pick_col(measures_df, "Table Name", "TableName", "Table")
m_name  = _pick_col(measures_df, "Measure Name", "MeasureName", "Name")
c_table = _pick_col(columns_df,  "Table Name", "TableName", "Table")
c_name  = _pick_col(columns_df,  "Column Name", "ColumnName", "Name")

print("Diagnostic info:")
print(f"  measures_df columns : {list(measures_df.columns)}")
print(f"  columns_df columns  : {list(columns_df.columns)}")
print(f"  Resolved measure key: ({m_table}, {m_name})")
print(f"  Resolved column key : ({c_table}, {c_name})")
print(f"  usage_df rows       : {len(usage_df)}  (0 = no PBIR reports scanned)")
print(f"  alive set size      : {len(alive)}")
print()

if not (m_table and m_name and c_table and c_name):
    print("❌ Could not resolve required columns. Aborting.")
else:
    measures_keys = set(zip(measures_df[m_table], measures_df[m_name]))
    columns_keys  = set(zip(columns_df[c_table],  columns_df[c_name]))

    if len(usage_df) == 0:
        # ── Fallback: no PBIR reports ─────────────────────────────────────────
        # A measure is "used in model" if any other measure's DAX calls it
        # (i.e. it appears as a child in the dependency graph).
        print("⚠️  No PBIR reports scanned — using model-graph-only detection.\n")
        used_as_child = set()
        for children in parent_to_children.values():
            used_as_child |= children
        # A measure is truly unused if nothing calls it AND it calls nothing
        # (pure orphan). Keep "helper measures" that call other measures.
        unused_measures_report = sorted(
            m for m in measures_keys
            if m not in used_as_child
        )
        unused_columns_report = sorted(columns_keys - used_as_child)
    else:
        # ── Full report-aware unused ──────────────────────────────────────────
        unused_measures_report = sorted(measures_keys - alive)
        unused_columns_report  = sorted(columns_keys  - alive)

    # ── Annotate unused measures: are they referenced by any other measure? ──
    rows = []
    for t, o in unused_measures_report:
        key = (t, o)
        # Who calls this measure?
        callers = child_to_parents.get(key, set())
        # What does this measure call?
        callees = parent_to_children.get(key, set())
        rows.append({
            "Type"               : "Measure",
            "Table"              : t,
            "Object"             : o,
            "Referenced By"      : ", ".join(f"{pt}[{po}]" for pt, po in sorted(callers)) or "—",
            "References"         : ", ".join(f"{ct}[{co}]" for ct, co in sorted(callees)) or "—",
            "Status"             : "Orphan (safe to delete)" if not callers and not callees
                                   else ("Helper measure — callers also unused" if callers else
                                         "Calls other objects but not used by any measure or report"),
        })

    for t, o in unused_columns_report:
        key = (t, o)
        callers = child_to_parents.get(key, set())
        rows.append({
            "Type"          : "Column",
            "Table"         : t,
            "Object"        : o,
            "Referenced By" : ", ".join(f"{pt}[{po}]" for pt, po in sorted(callers)) or "—",
            "References"    : "—",
            "Status"        : "Unused column" if not callers else "Called by unused measure only",
        })

    unused_df = pd.DataFrame(rows)

    print(f"Total measures in model : {len(measures_keys)}")
    print(f"Total columns  in model : {len(columns_keys)}")
    print(f"Unused measures         : {len(unused_measures_report)}")
    print(f"Unused columns          : {len(unused_columns_report)}")
    print()
    if len(unused_df) == 0:
        print("✅ No unused objects detected.")
    else:
        display(unused_df)


## 8. "Where is this object used?" — lineage lookup

Pass a table + object name to see every report/page/visual/filter where it is referenced
**plus** the model objects that depend on it (transitively).


In [ ]:
# Reverse adjacency for "what depends on me?" walks.
child_to_parents = {}
for parent, children in parent_to_children.items():
    for c in children:
        child_to_parents.setdefault(c, set()).add(parent)

def where_used(table: str, obj: str) -> dict:
    target = (table, obj)

    # 1) Direct report references (visuals/filters/CF)
    direct = usage_df[(usage_df["Table"] == table) & (usage_df["Object"] == obj)]

    # 2) Model dependents — measures whose DAX uses this object (closure upward)
    seen = set()
    stack = [target]
    while stack:
        n = stack.pop()
        for p in child_to_parents.get(n, ()):
            if p not in seen:
                seen.add(p)
                stack.append(p)

    # 3) Reports that reference any of those dependent measures (indirect usage)
    if seen:
        dep_keys = pd.DataFrame(list(seen), columns=["Table", "Object"])
        indirect = usage_df.merge(dep_keys, on=["Table", "Object"], how="inner")
    else:
        indirect = usage_df.iloc[0:0]

    return {
        "target": target,
        "direct_report_refs": direct.reset_index(drop=True),
        "model_dependents":   pd.DataFrame(sorted(seen), columns=["Table", "Object"]),
        "indirect_report_refs": indirect.reset_index(drop=True),
    }

# Example — change to a real (table, object) in your model.
example_target = next(iter(measures_keys), (None, None))
if example_target[0]:
    res = where_used(*example_target)
    print(f"Where used: {example_target}")
    print(f"  direct report references : {len(res['direct_report_refs'])}")
    print(f"  model dependents         : {len(res['model_dependents'])}")
    print(f"  indirect report refs     : {len(res['indirect_report_refs'])}")
    display(res["direct_report_refs"].head(20))


## 9. Tree view of measure dependencies

`labs.measure_dependency_tree` prints an indented tree for a chosen measure showing every
referenced measure / column / table.


In [ ]:
# Built-in tree printer (preferred). Falls back to a local printer if the function isn't present.
TARGET_MEASURE = measures_df["Measure Name"].iloc[0] if len(measures_df) else None
print(f"Target measure: {TARGET_MEASURE}\n")

if TARGET_MEASURE and hasattr(labs, "measure_dependency_tree"):
    labs.measure_dependency_tree(
        dataset=DATASET, workspace=WORKSPACE, measure_name=TARGET_MEASURE
    )
else:
    def print_tree(node, depth=0, visited=None):
        visited = visited or set()
        if node in visited:
            print("  " * depth + f"- {node} (cycle)")
            return
        visited = visited | {node}
        print("  " * depth + f"- {node[0]}[{node[1]}]")
        for child in sorted(parent_to_children.get(node, ())):
            print_tree(child, depth + 1, visited)

    if TARGET_MEASURE:
        # Find the (table, measure) tuple
        row = measures_df[measures_df["Measure Name"] == TARGET_MEASURE].iloc[0]
        print_tree((row["Table Name"], row["Measure Name"]))


## 10. Export model + report documentation

Writes a single Excel workbook (one sheet per topic) **and** a Markdown summary you can drop
into a wiki. Output goes to `EXPORT_DIR`. If a Lakehouse isn't mounted, the cell falls back to
the notebook's local working directory.


In [ ]:
import os, datetime
out_dir = EXPORT_DIR
try:
    os.makedirs(out_dir, exist_ok=True)
except Exception:
    out_dir = os.path.abspath("./model_docs")
    os.makedirs(out_dir, exist_ok=True)

stamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
xlsx_path = os.path.join(out_dir, f"{DATASET}_documentation_{stamp}.xlsx")
md_path   = os.path.join(out_dir, f"{DATASET}_documentation_{stamp}.md")

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
    tables_df.to_excel(xw,        sheet_name="Tables",         index=False)
    columns_df.to_excel(xw,       sheet_name="Columns",        index=False)
    measures_df.to_excel(xw,      sheet_name="Measures",       index=False)
    relationships_df.to_excel(xw, sheet_name="Relationships",  index=False)
    deps_df.to_excel(xw,          sheet_name="MeasureDeps",    index=False)
    usage_df.to_excel(xw,         sheet_name="ReportUsage",    index=False)
    unused_df.to_excel(xw,        sheet_name="ReportUnused",   index=False)
    if len(model_unused_df):
        model_unused_df.to_excel(xw, sheet_name="ModelUnused",  index=False)

# Markdown summary
lines = [
    f"# {DATASET} — Documentation",
    f"_Workspace: **{WORKSPACE}** · Generated: {stamp} UTC_",
    "",
    f"- Tables: **{len(tables_df)}**",
    f"- Columns: **{len(columns_df)}**",
    f"- Measures: **{len(measures_df)}**",
    f"- Relationships: **{len(relationships_df)}**",
    f"- Reports scanned: **{len(report_names)}**",
    f"- Object references in reports: **{len(usage_df)}**",
    f"- Report-unused measures: **{len(unused_measures_report)}**",
    f"- Report-unused columns : **{len(unused_columns_report)}**",
    "",
    "## Reports",
    *[f"- {r}" for r in report_names],
    "",
    "## Top unused measures (sample)",
    *[f"- `{t}`[{o}]" for t, o in unused_measures_report[:25]],
    "",
    "## Top unused columns (sample)",
    *[f"- `{t}`[{o}]" for t, o in unused_columns_report[:25]],
]
with open(md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"Excel : {xlsx_path}")
print(f"MD    : {md_path}")
